<a href="https://colab.research.google.com/github/jinofedolin/3-wheel-omni-directional-robot/blob/main/3_wheel_omni_directional_robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# CONTINUOUS 3-WHEEL OMNI-DIRECTIONAL ROBOT SIMULATOR
# GOOGLE COLAB
#
# Live controls:
#   omega1, omega2, omega3
#
# Robot continuously moves while sliders are changed.
# ================================================================

from IPython.display import HTML, display

html_code = r'''
<!DOCTYPE html>

<html>

<head>

<style>

body {
    font-family: Arial, sans-serif;
    background: white;
    margin: 0;
    padding: 10px;
}

.container {
    width: 100%;
    max-width: 1100px;
    margin: auto;
}

h2 {
    text-align: center;
    color: #1565C0;
    margin: 5px;
}

#canvas {
    display: block;
    margin: auto;
    border: 2px solid #333;
    background: white;
}

.controls {
    display: flex;
    justify-content: center;
    flex-wrap: wrap;
    gap: 15px;
    margin-top: 10px;
}

.control {
    padding: 10px;
    border: 1px solid #aaa;
    border-radius: 8px;
    background: #f5f5f5;
    width: 250px;
}

.control label {
    font-weight: bold;
}

input[type=range] {
    width: 100%;
}

button {
    padding: 10px 20px;
    margin: 5px;
    font-size: 15px;
    border-radius: 6px;
    border: none;
    cursor: pointer;
}

.start {
    background: #2E7D32;
    color: white;
}

.stop {
    background: #C62828;
    color: white;
}

.reset {
    background: #1565C0;
    color: white;
}

.values {
    text-align: center;
    font-size: 16px;
    margin-top: 10px;
    padding: 10px;
    border: 1px solid #aaa;
    border-radius: 8px;
    background: #fafafa;
}

</style>

</head>


<body>

<div class="container">

<h2>
THREE-WHEEL OMNI-DIRECTIONAL ROBOT
</h2>

<div style="text-align:center;">
Continuous Real-Time Kinematic Simulation
</div>


<canvas id="canvas"
        width="900"
        height="650">
</canvas>


<div class="controls">


<div class="control">

<label>
ω1:
<span id="w1value">0.0</span>
rad/s
</label>

<input
type="range"
id="w1"
min="-20"
max="20"
step="0.1"
value="0">

</div>



<div class="control">

<label>
ω2:
<span id="w2value">0.0</span>
rad/s
</label>

<input
type="range"
id="w2"
min="-20"
max="20"
step="0.1"
value="0">

</div>



<div class="control">

<label>
ω3:
<span id="w3value">0.0</span>
rad/s
</label>

<input
type="range"
id="w3"
min="-20"
max="20"
step="0.1"
value="0">

</div>

</div>



<div style="text-align:center;">

<button class="start" onclick="startRobot()">
START
</button>

<button class="stop" onclick="stopRobot()">
STOP
</button>

<button class="reset" onclick="resetRobot()">
RESET
</button>

</div>



<div class="values">

<b>Robot Velocity</b>

<br>

Vx =
<span id="vx">0.000</span>
m/s

&nbsp;&nbsp;&nbsp;

Vy =
<span id="vy">0.000</span>
m/s

&nbsp;&nbsp;&nbsp;

Ω =
<span id="omega">0.000</span>
rad/s

<br><br>

<b>Robot Pose</b>

<br>

X =
<span id="xpos">0.000</span>
m

&nbsp;&nbsp;&nbsp;

Y =
<span id="ypos">0.000</span>
m

&nbsp;&nbsp;&nbsp;

θ =
<span id="theta">0.00</span>
°

</div>


</div>



<script>

// ============================================================
// CANVAS
// ============================================================

const canvas =
document.getElementById("canvas");

const ctx =
canvas.getContext("2d");


// ============================================================
// ROBOT PARAMETERS
// ============================================================

// Wheel radius
const r = 0.05;

// Distance from center to wheel
const L = 0.20;


// Robot graphical size
const robotRadius = 65;


// Scale
// pixels per meter
const scale = 150;


// ============================================================
// ROBOT STATE
// ============================================================

let x = 0;
let y = 0;
let theta = 0;


// Wheel rotation angles
let wheelRot1 = 0;
let wheelRot2 = 0;
let wheelRot3 = 0;


// Simulation state
let running = true;


// ============================================================
// TRAJECTORY
// ============================================================

let trajectory = [];


// ============================================================
// WHEEL SLIDERS
// ============================================================

const w1 =
document.getElementById("w1");

const w2 =
document.getElementById("w2");

const w3 =
document.getElementById("w3");


// ============================================================
// UPDATE SLIDER VALUES
// ============================================================

w1.oninput = function()
{
    document.getElementById("w1value").innerHTML =
    parseFloat(this.value).toFixed(1);
};


w2.oninput = function()
{
    document.getElementById("w2value").innerHTML =
    parseFloat(this.value).toFixed(1);
};


w3.oninput = function()
{
    document.getElementById("w3value").innerHTML =
    parseFloat(this.value).toFixed(1);
};


// ============================================================
// FORWARD KINEMATICS
// ============================================================

function calculateVelocity()
{

    let omega1 =
    parseFloat(w1.value);

    let omega2 =
    parseFloat(w2.value);

    let omega3 =
    parseFloat(w3.value);


    // --------------------------------------------------------
    // Omni-wheel inverse/forward kinematics
    // --------------------------------------------------------

    let Vx =
    (r / Math.sqrt(3))
    *
    (omega3 - omega2);


    let Vy =
    (r / 3)
    *
    (2*omega1 - omega2 - omega3);


    let Omega =
    (r / (3*L))
    *
    (omega1 + omega2 + omega3);


    return {
        Vx: Vx,
        Vy: Vy,
        Omega: Omega
    };
}


// ============================================================
// WORLD TO SCREEN
// ============================================================

function worldX(x)
{
    return canvas.width/2 + x*scale;
}


function worldY(y)
{
    return canvas.height/2 - y*scale;
}


// ============================================================
// DRAW GRID
// ============================================================

function drawGrid()
{

    ctx.strokeStyle = "#dddddd";
    ctx.lineWidth = 1;


    // Vertical grid
    for(
        let X = canvas.width/2 % scale;
        X < canvas.width;
        X += scale
    )
    {

        ctx.beginPath();

        ctx.moveTo(X,0);

        ctx.lineTo(X,canvas.height);

        ctx.stroke();

    }


    // Horizontal grid
    for(
        let Y = canvas.height/2 % scale;
        Y < canvas.height;
        Y += scale
    )
    {

        ctx.beginPath();

        ctx.moveTo(0,Y);

        ctx.lineTo(canvas.width,Y);

        ctx.stroke();

    }


    // X axis

    ctx.strokeStyle = "#555";
    ctx.lineWidth = 2;

    ctx.beginPath();

    ctx.moveTo(0,canvas.height/2);

    ctx.lineTo(canvas.width,canvas.height/2);

    ctx.stroke();


    // Y axis

    ctx.beginPath();

    ctx.moveTo(canvas.width/2,0);

    ctx.lineTo(canvas.width/2,canvas.height);

    ctx.stroke();


    // Axis labels

    ctx.fillStyle = "#333";

    ctx.font = "16px Arial";

    ctx.fillText(
        "X",
        canvas.width-25,
        canvas.height/2-10
    );

    ctx.fillText(
        "Y",
        canvas.width/2+10,
        20
    );

}


// ============================================================
// DRAW TRAJECTORY
// ============================================================

function drawTrajectory()
{

    if(trajectory.length < 2)
        return;


    ctx.strokeStyle = "#1565C0";

    ctx.lineWidth = 3;

    ctx.beginPath();


    let p =
    trajectory[0];


    ctx.moveTo(
        worldX(p.x),
        worldY(p.y)
    );


    for(
        let i=1;
        i<trajectory.length;
        i++
    )
    {

        p = trajectory[i];

        ctx.lineTo(
            worldX(p.x),
            worldY(p.y)
        );

    }


    ctx.stroke();

}


// ============================================================
// DRAW ROBOT
// ============================================================

function drawRobot()
{

    let cx = worldX(x);
    let cy = worldY(y);


    ctx.save();

    ctx.translate(cx,cy);

    ctx.rotate(-theta);


    // --------------------------------------------------------
    // Robot body
    // --------------------------------------------------------

    ctx.beginPath();

    ctx.arc(
        0,
        0,
        robotRadius,
        0,
        2*Math.PI
    );

    ctx.fillStyle = "#90CAF9";

    ctx.fill();

    ctx.strokeStyle = "#1565C0";

    ctx.lineWidth = 4;

    ctx.stroke();


    // --------------------------------------------------------
    // Heading arrow
    // --------------------------------------------------------

    ctx.beginPath();

    ctx.moveTo(0,0);

    ctx.lineTo(
        robotRadius+40,
        0
    );

    ctx.strokeStyle = "#D32F2F";

    ctx.lineWidth = 5;

    ctx.stroke();


    // Arrow head

    ctx.beginPath();

    ctx.moveTo(
        robotRadius+40,
        0
    );

    ctx.lineTo(
        robotRadius+20,
        -10
    );

    ctx.lineTo(
        robotRadius+20,
        10
    );

    ctx.closePath();

    ctx.fillStyle = "#D32F2F";

    ctx.fill();


    // --------------------------------------------------------
    // Wheel locations
    // --------------------------------------------------------

    let angles =
    [
        Math.PI/2,
        7*Math.PI/6,
        11*Math.PI/6
    ];


    let colors =
    [
        "#E53935",
        "#1E88E5",
        "#43A047"
    ];


    let labels =
    [
        "W1",
        "W2",
        "W3"
    ];


    let wheelRot =
    [
        wheelRot1,
        wheelRot2,
        wheelRot3
    ];


    for(
        let i=0;
        i<3;
        i++
    )
    {

        let a = angles[i];


        let wx =
        L*scale*Math.cos(a);

        let wy =
        -L*scale*Math.sin(a);


        ctx.save();


        ctx.translate(
            wx,
            wy
        );


        // wheel body

        ctx.fillStyle =
        colors[i];


        ctx.beginPath();

        ctx.arc(
            0,
            0,
            18,
            0,
            2*Math.PI
        );

        ctx.fill();


        ctx.strokeStyle = "#222";

        ctx.lineWidth = 2;

        ctx.stroke();


        // wheel rotation indicator

        ctx.rotate(
            wheelRot[i]
        );


        ctx.beginPath();

        ctx.moveTo(
            -13,
            0
        );

        ctx.lineTo(
            13,
            0
        );

        ctx.strokeStyle =
        "white";

        ctx.lineWidth = 3;

        ctx.stroke();


        ctx.restore();


        // wheel label

        ctx.fillStyle = "#111";

        ctx.font =
        "bold 15px Arial";


        ctx.fillText(
            labels[i],
            wx+20,
            wy
        );

    }


    // Robot center

    ctx.beginPath();

    ctx.arc(
        0,
        0,
        6,
        0,
        2*Math.PI
    );

    ctx.fillStyle =
    "#111";

    ctx.fill();


    ctx.restore();

}


// ============================================================
// UPDATE ROBOT
// ============================================================

function updateRobot(dt)
{

    let velocity =
    calculateVelocity();


    let Vx =
    velocity.Vx;

    let Vy =
    velocity.Vy;

    let Omega =
    velocity.Omega;


    // --------------------------------------------------------
    // Body frame -> world frame
    // --------------------------------------------------------

    let xDot =
    Vx*Math.cos(theta)
    -
    Vy*Math.sin(theta);


    let yDot =
    Vx*Math.sin(theta)
    +
    Vy*Math.cos(theta);


    // --------------------------------------------------------
    // Integrate
    // --------------------------------------------------------

    x += xDot*dt;

    y += yDot*dt;

    theta += Omega*dt;


    // --------------------------------------------------------
    // Wheel animation
    // --------------------------------------------------------

    wheelRot1 +=
    parseFloat(w1.value)*dt;

    wheelRot2 +=
    parseFloat(w2.value)*dt;

    wheelRot3 +=
    parseFloat(w3.value)*dt;


    // --------------------------------------------------------
    // Store trajectory
    // --------------------------------------------------------

    trajectory.push({
        x:x,
        y:y
    });


    // Limit trajectory memory

    if(
        trajectory.length > 3000
    )
    {
        trajectory.shift();
    }


    // --------------------------------------------------------
    // Update numerical display
    // --------------------------------------------------------

    document.getElementById("vx")
    .innerHTML =
    Vx.toFixed(3);


    document.getElementById("vy")
    .innerHTML =
    Vy.toFixed(3);


    document.getElementById("omega")
    .innerHTML =
    Omega.toFixed(3);


    document.getElementById("xpos")
    .innerHTML =
    x.toFixed(3);


    document.getElementById("ypos")
    .innerHTML =
    y.toFixed(3);


    document.getElementById("theta")
    .innerHTML =
    (theta*180/Math.PI).toFixed(2);

}


// ============================================================
// DRAW EVERYTHING
// ============================================================

function draw()
{

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    drawGrid();

    drawTrajectory();

    drawRobot();

}


// ============================================================
// ANIMATION LOOP
// ============================================================

let lastTime =
performance.now();


function animationLoop(currentTime)
{

    let dt =
    (currentTime-lastTime)/1000;


    lastTime =
    currentTime;


    // Limit dt
    if(dt > 0.05)
        dt = 0.05;


    if(running)
    {

        updateRobot(dt);

    }


    draw();


    requestAnimationFrame(
        animationLoop
    );

}


// ============================================================
// START
// ============================================================

function startRobot()
{

    running = true;

}


// ============================================================
// STOP
// ============================================================

function stopRobot()
{

    running = false;

}


// ============================================================
// RESET
// ============================================================

function resetRobot()
{

    x = 0;

    y = 0;

    theta = 0;


    wheelRot1 = 0;

    wheelRot2 = 0;

    wheelRot3 = 0;


    trajectory = [];


    trajectory.push({
        x:0,
        y:0
    });


    draw();

}


// ============================================================
// INITIALIZE
// ============================================================

resetRobot();


// Start continuous animation

requestAnimationFrame(
    animationLoop
);

</script>

</body>

</html>
'''

display(HTML(html_code))